In [2]:
import os
from dotenv import load_dotenv

env_path = r"C:\Users\ykost\Jupyter\test2\CAP_3764_2026_Spring_Team_3\.env"
load_dotenv(env_path)

print("API key loaded:", bool(os.getenv("NEWSAPI_KEY")))

API key loaded: True


In [3]:
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("NEWSAPI_KEY")
assert API_KEY, "NEWSAPI_KEY not loaded"

DAYS_BACK = 21
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=DAYS_BACK)

START = start_date.strftime("%Y-%m-%d")
END = end_date.strftime("%Y-%m-%d")

print("Using date range:", START, "to", END)


Using date range: 2026-01-20 to 2026-02-10


In [6]:
def fetch_articles(ticker, max_results=100):
    params = {
        "q": QUERIES[ticker],
        "from": START,
        "to": END,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": max_results,  # <= 100 for free tier
        "page": 1,
    }
    headers = {"X-Api-Key": API_KEY}

    r = requests.get(NEWSAPI_ENDPOINT, params=params, headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"{ticker} NewsAPI error {r.status_code}: {r.text}")

    rows = []
    articles = r.json().get("articles", [])

    for a in articles:
        published_at = a.get("publishedAt")
        if not published_at:
            continue

        dt = datetime.fromisoformat(published_at.replace("Z", "+00:00"))
        rows.append({
            "ticker": ticker,
            "date": dt.strftime("%Y-%m-%d"),
            "headline": (a.get("title") or "").strip(),
            "content": (a.get("content") or a.get("description") or "").strip(),
            "source": ((a.get("source") or {}).get("name") or "").strip(),
            "url": (a.get("url") or "").strip(),
        })

    return rows

In [9]:
all_rows = []
for t in ["AAPL", "NVDA"]:
    print(f"Fetching {t}...")
    rows = fetch_articles(ticker=t, max_results=100)
    print(f"  Raw pulled: {len(rows)}")
    all_rows.extend(rows)

articles_raw = pd.DataFrame(all_rows)
articles_raw.head()


Fetching AAPL...
  Raw pulled: 97
Fetching NVDA...
  Raw pulled: 97


,ticker,date,headline,content,source,url
0,AAPL,2026-02-09,Affordable 5G fuels India smartphone market de...,"India's smartphone market declined 1% in 2025,...",Business Standard,https://www.business-standard.com/technology/t...
1,AAPL,2026-02-09,How to Backup an iPhone Without iCloud?,Image Credit YouTube\r\nBacking up your iPhone...,Helpdeskgeek.com,https://helpdeskgeek.com/how-to-backup-an-ipho...
2,AAPL,2026-02-09,Hundreds in Brooklyn without power as temperat...,Hundreds of Brooklyn residents will have no po...,New York Post,https://nypost.com/2026/02/08/us-news/hundreds...
3,AAPL,2026-02-09,"‘One of the worst, EVER!’ Trump torches Super ...",Bad Bunny performs during Super Bowl LX in San...,Wnd.com,https://www.wnd.com/2026/02/one-worst-ever-tru...
4,AAPL,2026-02-09,Trump Calls Bad Bunny's Super Bowl Halftime Sh...,While Bad Bunny was expressing love during his...,Hollywood Reporter,http://www.hollywoodreporter.com/music/music-n...


In [10]:
articles = articles_raw.copy()

# Standardize date and drop missing ticker/date
articles["date"] = pd.to_datetime(articles["date"], errors="coerce").dt.strftime("%Y-%m-%d")
articles = articles.dropna(subset=["ticker", "date"])
articles = articles[articles["ticker"].astype(str).str.len() > 0]

# Remove duplicates
if "url" in articles.columns:
    articles = articles.drop_duplicates(subset=["url"], keep="first")
articles = articles.drop_duplicates(subset=["ticker", "date", "headline"], keep="first")

# Keep only required columns
articles = articles[["ticker", "date", "headline", "content", "source"]]

print("Counts after cleaning:")
print(articles["ticker"].value_counts())

articles.head()

Counts after cleaning:
ticker
AAPL    96
NVDA    92
Name: count, dtype: int64


,ticker,date,headline,content,source
0,AAPL,2026-02-09,Affordable 5G fuels India smartphone market de...,"India's smartphone market declined 1% in 2025,...",Business Standard
1,AAPL,2026-02-09,How to Backup an iPhone Without iCloud?,Image Credit YouTube\r\nBacking up your iPhone...,Helpdeskgeek.com
2,AAPL,2026-02-09,Hundreds in Brooklyn without power as temperat...,Hundreds of Brooklyn residents will have no po...,New York Post
3,AAPL,2026-02-09,"‘One of the worst, EVER!’ Trump torches Super ...",Bad Bunny performs during Super Bowl LX in San...,Wnd.com
4,AAPL,2026-02-09,Trump Calls Bad Bunny's Super Bowl Halftime Sh...,While Bad Bunny was expressing love during his...,Hollywood Reporter


In [11]:
articles.to_csv("articles.csv", index=False, encoding="utf-8")
print("Saved articles.csv")

Saved articles.csv


In [13]:
import yfinance as yf

In [14]:
TICKERS = ["AAPL", "NVDA"]

# yfinance end is often exclusive, add 1 day buffer
end_plus = (datetime.fromisoformat(END) + timedelta(days=1)).strftime("%Y-%m-%d")

data = yf.download(
    tickers=" ".join(TICKERS),
    start=START,
    end=end_plus,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    progress=False,
)

rows = []
for t in TICKERS:
    sub = data[t].reset_index()
    sub["ticker"] = t
    rows.append(sub)

prices = pd.concat(rows, ignore_index=True)

prices = prices.rename(columns={
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume",
})

prices["date"] = pd.to_datetime(prices["date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Drop rows missing ticker/date (cleaning requirement)
prices = prices.dropna(subset=["ticker", "date"])
prices = prices.drop_duplicates(subset=["ticker", "date"], keep="first")

prices = prices[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]].copy()

print("Price rows per ticker:")
print(prices["ticker"].value_counts())

prices.head()

Price rows per ticker:
ticker
AAPL    15
NVDA    15
Name: count, dtype: int64


Price,ticker,date,open,high,low,close,adj_close,volume
0,AAPL,2026-01-20,252.729996,254.789993,243.419998,246.699997,246.469376,80267500
1,AAPL,2026-01-21,248.699997,251.559998,245.179993,247.649994,247.418488,54641700
2,AAPL,2026-01-22,249.199997,251.000000,248.149994,248.350006,248.117844,39708300
3,AAPL,2026-01-23,247.320007,249.410004,244.679993,248.039993,247.808121,41689000
4,AAPL,2026-01-26,251.479996,256.559998,249.800003,255.410004,255.171234,55969200


In [15]:
prices.to_csv("prices.csv", index=False, encoding="utf-8")
print("Saved prices.csv")

Saved prices.csv
